# Hey Emma — Custom Wake Word Training

Trainiert ein openWakeWord-Modell für **"Hey Emma"** mit synthetischen Stimmen (Piper TTS).

**Voraussetzungen:** Google Colab mit GPU-Runtime (T4 reicht, beschleunigt TTS-Generierung).

**Ergebnis:** `hey_emma.onnx` — bereit für den Einsatz in der Hey Emma App.

## 1. Environment Setup

In [ ]:
# Install openWakeWord with training dependencies
!pip install openwakeword[train] onnx onnxruntime tflite-runtime
!pip install datasets huggingface_hub

In [ ]:
# Clone piper-sample-generator for synthetic TTS
!git clone https://github.com/dscripka/piper-sample-generator.git

# Download the Piper TTS model (LibriTTS, ~904 speakers)
!cd piper-sample-generator && python -m pip install -e .

import os
PIPER_PATH = os.path.abspath("piper-sample-generator")
print(f"Piper path: {PIPER_PATH}")

## 2. Download Training Data

- **Negative features:** ~2000h ACAV100M (vorberechnete Audio-Embeddings)
- **Validation set:** ~11h für False-Positive-Rate-Messung
- **Room Impulse Responses:** MIT RIRs für Augmentierung
- **Background noise:** Umgebungsgeräusche für Augmentierung

In [ ]:
import os
os.makedirs("training_data", exist_ok=True)
os.chdir("training_data")

In [ ]:
# Download pre-computed negative features from HuggingFace (~2000 hrs)
from huggingface_hub import hf_hub_download

negative_features_path = hf_hub_download(
    repo_id="dscripka/openwakeword_features",
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    repo_type="dataset",
    local_dir=".",
)
print(f"Negative features: {negative_features_path}")

In [ ]:
# Download false-positive validation set
validation_path = hf_hub_download(
    repo_id="dscripka/openwakeword_features",
    filename="validation_set_features.npy",
    repo_type="dataset",
    local_dir=".",
)
print(f"Validation features: {validation_path}")

In [ ]:
# Download MIT Room Impulse Responses
!wget -q https://www.openslr.org/resources/28/rirs_noises.zip
!unzip -q -o rirs_noises.zip -d mit_rirs
!rm rirs_noises.zip
print("MIT RIRs downloaded.")

In [ ]:
# Download background noise dataset (FreeSound clips from openWakeWord)
!wget -q https://www.openslr.org/resources/17/musan.tar.gz
!tar -xzf musan.tar.gz
!rm musan.tar.gz
print("Background noise (MUSAN) downloaded.")

In [ ]:
os.chdir("..")
print(f"Working directory: {os.getcwd()}")

## 3. Training Config

Die Config wird direkt hier erstellt — keine externe Datei nötig.

In [ ]:
import yaml

config = {
    "model_name": "hey_emma",
    "target_phrase": ["hey emma"],

    # Phonetically similar phrases to suppress false positives
    "custom_negative_phrases": [
        "hey anna",
        "hey ella",
        "hey eva",
        "hey ever",
        "hey oma",
        "hey irma",
        "hey mama",
        "hey lemma",
        "hey thema",
    ],

    # Synthetic data generation
    "n_samples": 50000,
    "n_samples_val": 5000,
    "tts_batch_size": 100,

    "piper_sample_generator_path": PIPER_PATH,
    "output_dir": "./hey_emma_model",

    # Augmentation
    "augmentation_batch_size": 16,
    "augmentation_rounds": 2,
    "rir_paths": ["./training_data/mit_rirs/RIRS_NOISES/simulated_rirs"],
    "background_paths": ["./training_data/musan/noise"],
    "background_paths_duplication_rate": [1],

    # Pre-computed negative features
    "feature_data_files": {
        "ACAV100M_sample": "./training_data/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    },
    "false_positive_validation_data_path": "./training_data/validation_set_features.npy",

    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50,
    },

    # Model architecture
    "model_type": "dnn",
    "layer_size": 32,

    # Training parameters
    "steps": 50000,
    "max_negative_weight": 1500,
    "target_false_positives_per_hour": 0.2,
}

with open("hey_emma.yml", "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Config written to hey_emma.yml")
print(yaml.dump(config, default_flow_style=False, sort_keys=False))

## 4. Generate Synthetic Clips

Piper TTS erzeugt 50.000 Varianten von "Hey Emma" mit ~904 verschiedenen Stimmen.
Auf einer T4-GPU dauert das ca. 10–20 Minuten.

In [ ]:
!python -m openwakeword.train \
    --training_config hey_emma.yml \
    --generate_clips

## 5. Augment Clips

Wendet Room Impulse Responses und Hintergrundgeräusche an.
2 Augmentation-Runden verdoppeln die Datenvielfalt.

In [ ]:
!python -m openwakeword.train \
    --training_config hey_emma.yml \
    --augment_clips

## 6. Train Model

Trainiert ein kleines DNN (2x 32 Units) auf den Audio-Embeddings.
Ziel: ≤ 0.2 False Positives pro Stunde.

In [ ]:
!python -m openwakeword.train \
    --training_config hey_emma.yml \
    --train_model

## 7. Export to ONNX + TFLite

In [ ]:
!python -m openwakeword.train \
    --training_config hey_emma.yml \
    --convert_to_tflite

In [ ]:
import glob

models = glob.glob("hey_emma_model/**/*.onnx", recursive=True)
models += glob.glob("hey_emma_model/**/*.tflite", recursive=True)

print("Exportierte Modelle:")
for m in sorted(models):
    size_kb = os.path.getsize(m) / 1024
    print(f"  {m} ({size_kb:.0f} KB)")

## 8. Quick Test

Schnelltest mit synthetischem Audio, um zu prüfen ob das Modell reagiert.

In [ ]:
import numpy as np
from openwakeword.model import Model

# Find the exported ONNX model
onnx_models = glob.glob("hey_emma_model/**/*.onnx", recursive=True)
model_path = [m for m in onnx_models if "hey_emma" in m and "embedding" not in m and "melspec" not in m]

if model_path:
    print(f"Testing model: {model_path[0]}")
    oww = Model(wakeword_models=[model_path[0]], inference_framework="onnx")
    print(f"Loaded models: {list(oww.models.keys())}")

    # Test with a positive clip from the generated validation set
    val_clips = glob.glob("hey_emma_model/**/positive_val/*.wav", recursive=True)
    if val_clips:
        import wave
        with wave.open(val_clips[0], "rb") as wf:
            audio = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16)

        # Feed in 1280-sample chunks
        max_score = 0.0
        for i in range(0, len(audio) - 1280, 1280):
            chunk = audio[i:i+1280]
            prediction = oww.predict(chunk)
            for name, score in prediction.items():
                max_score = max(max_score, score)

        print(f"Max score on positive clip: {max_score:.3f}")
        print("PASS" if max_score > 0.5 else "WARN: score below threshold, consider retraining")
    else:
        print("No validation clips found for testing.")
else:
    print("ERROR: No ONNX model found in output directory.")

## 9. Download Model

Das fertige Modell herunterladen und in `resources/` des Hey Emma Projekts ablegen.

In `.env` setzen:
```
OPENWAKEWORD_MODEL_PATH=hey_emma.onnx
OPENWAKEWORD_KEYWORD=Hey-Emma
```

In [ ]:
# In Colab: Download via browser
try:
    from google.colab import files
    if model_path:
        files.download(model_path[0])
        print("Download gestartet.")
except ImportError:
    if model_path:
        print(f"Modell bereit: {model_path[0]}")
        print("Kopiere die Datei nach resources/hey_emma.onnx im Projekt.")